# PROMPT ENGINEERING

## SETTING UP THE ENVIRONMENT

In [1]:
!pip install -q transformers accelerate sentencepiece

Importing the libraries:

In [ ]:
import re
import json
import torch

from spacy        import displacy
from transformers import AutoTokenizer, AutoModelForCausalLM

## PROMPT ENGINEERING

In [ ]:
model = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer       .from_pretrained(model)
model     = AutoModelForCausalLM.from_pretrained(
    model,
    dtype     =torch.float16,
    device_map="auto"       ,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


total_params = count_parameters(model)

print(f"Total parameters: {total_params:,}")

Total parameters: 7,615,616,512


In [5]:
SYSTEM_INSTRUCTIONS = """
Você é um anotador de entidades nomeadas (NER) em Português.
Tarefa: dado um texto em português, retorna APENAS um JSON contendo uma lista chamada "entities".
Cada entidade deve ter os campos:
 - "text": a substring exata do texto (sem alterações),
 - "label": uma de ["PER","LOC","ORG"],
 - "start": índice de caractere do início (0-based),
 - "end": índice do final (exclusivo, i.e. Python slicing compatible).
Regras:
 - Retorne apenas JSON válido (não inclua explicações ou texto adicional).
 - Retorne apenas o JSON entre os {}, sem nenhum texto complementar como o prompt.
 - Inclua todas as ocorrências de entidades PER, LOC e ORG.
 - Não invente entidades.
 - Mantenha a ordem original das entidades como aparecem no texto.
"""

EXAMPLE_1_TEXT = "João e Maria viajaram de São Paulo para a Universidade de Brasília."
EXAMPLE_1_JSON = {
    "entities": [
        {"text": "João"                    , "label": "PER", "start": EXAMPLE_1_TEXT.index("João"                    ), "end": EXAMPLE_1_TEXT.index("João"                    ) + len("João"                    )},
        {"text": "Maria"                   , "label": "PER", "start": EXAMPLE_1_TEXT.index("Maria"                   ), "end": EXAMPLE_1_TEXT.index("Maria"                   ) + len("Maria"                   )},
        {"text": "São Paulo"               , "label": "LOC", "start": EXAMPLE_1_TEXT.index("São Paulo"               ), "end": EXAMPLE_1_TEXT.index("São Paulo"               ) + len("São Paulo"               )},
        {"text": "Universidade de Brasília", "label": "ORG", "start": EXAMPLE_1_TEXT.index("Universidade de Brasília"), "end": EXAMPLE_1_TEXT.index("Universidade de Brasília") + len("Universidade de Brasília")}
    ]
}

EXAMPLE_2_TEXT = "A Microsoft abriu um escritório no Rio de Janeiro."
EXAMPLE_2_JSON = {
    "entities": [
        {"text": "Microsoft"     , "label": "ORG", "start": EXAMPLE_2_TEXT.index("Microsoft"     ), "end": EXAMPLE_2_TEXT.index("Microsoft"     ) + len("Microsoft"     )},
        {"text": "Rio de Janeiro", "label": "LOC", "start": EXAMPLE_2_TEXT.index("Rio de Janeiro"), "end": EXAMPLE_2_TEXT.index("Rio de Janeiro") + len("Rio de Janeiro")}
    ]
}

In [6]:
def build_prompt(text):
    parts = [
        "System instructions:",
        SYSTEM_INSTRUCTIONS.strip(),
        "\nExemplo 1:\nTexto:\n\"\"\"\n" + EXAMPLE_1_TEXT + "\n\"\"\"\nSaída JSON:\n" + json.dumps(EXAMPLE_1_JSON, ensure_ascii=False, indent=2),
        "\nExemplo 2:\nTexto:\n\"\"\"\n" + EXAMPLE_2_TEXT + "\n\"\"\"\nSaída JSON:\n" + json.dumps(EXAMPLE_2_JSON, ensure_ascii=False, indent=2),
        "\nAgora, anote o seguinte texto (apenas devolva o JSON):\nTexto:\n\"\"\"\n"  + text + "\n\"\"\"\nSaída JSON:"
    ]

    return "\n\n".join(parts)


def model_generate(prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample  =False,
        temperature=1.0  ,
    )

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [7]:
def recalculate_indices(text, entities):
    updated = []

    for ent in entities:
        term = ent["text"]

        start = text.find(term)
        if start == -1:
            raise ValueError(f"Term '{term}' not found in text.")

        end = start + len(term)

        updated.append({
            "text"  : term        ,
            "label" : ent["label"],
            "start" : start       ,
            "end"   : end
        })

        text = text[:start] + (" " * len(term)) + text[end:]

    return updated

In [8]:
def extrair_ultimo_json(texto):
    idx = texto.rfind("Saída JSON:")
    if idx == -1:
        return None

    trecho = texto[idx + len("Saída JSON:"):].strip()

    count  = 0
    inicio = None
    for i, c in enumerate(trecho):
        if c == '{':
            if inicio is None:
                inicio = i
            count += 1
        elif c == '}':
            count -= 1

            if count == 0 and inicio is not None:
                return json.loads(
                    trecho[inicio:i + 1]
                )

    return None

In [9]:
def ner(example):
    prompt = build_prompt  (example)
    output = model_generate(prompt )

    pattern = r'```json\s*({[\s\S]*?\]\s*})\s*```'
    match_  = re.search(pattern, output)

    if match_:
        json_block = match_.group(1)
        json_block = json  .loads(json_block)
    else:
        json_block = extrair_ultimo_json(output)

        if not json_block:
            print("JSON não encontrado.")
            return

    return recalculate_indices(example, json_block["entities"])


def llm_json_to_displacy(text, llm_json):
    docs = [{
        "text" : text,
        "ents" : [
            {
                "start" : ent["start"],
                "end"   : ent["end"  ],
                "label" : ent["label"]
            }
            for ent in llm_json
        ],
    }]

    colors = {
        "PER": "linear-gradient(90deg, #999999, #cccccc)",
        "LOC": "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
        "ORG": "linear-gradient(90deg, #ffcc70, #ff9a3c)",
    }
    options = {"ents": ["PER", "LOC", "ORG"], "colors": colors}

    return docs, options

## TESTS

In [ ]:
text = "João encontrou Maria ontem à noite."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
text = "Estou indo para João Pessoa amanhã cedo."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "A Google lançou um novo modelo de IA."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
text = "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

In [ ]:
%%time

text = "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."


docs, options = llm_json_to_displacy(text, ner(text))

displacy.render(docs, style="ent", manual=True, options=options, jupyter=True)

CPU times: user 2min, sys: 2.11 s, total: 2min 2s
Wall time: 2min 2s
